<a href="https://colab.research.google.com/github/kumarpal1107/pythonbasics/blob/main/Sourcecode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Drop unwanted index column
if "unnamed: 0" in df.columns:
    df = df.drop(columns=["unnamed: 0"])

# Standardize column names
df.rename(columns={
    "crypto_name": "symbol",
    "marketcap": "market_cap"
}, inplace=True)

print("Columns after cleaning:", df.columns.tolist())

# ------------------------------------------
# Basic Info & Summary
# ------------------------------------------
print("\n--- Dataset Info ---")
print(df.info())

print("\n--- Summary Statistics ---")
print(df.describe())

print("\n--- Missing Values ---")
print(df.isna().sum())

# ------------------------------------------
# Unique Cryptocurrencies
# ------------------------------------------
if "symbol" in df.columns:
    print("\nUnique cryptocurrencies (symbols):", df["symbol"].nunique())
    print("Symbols:", df["symbol"].unique())
else:
    print("\nERROR: No 'symbol' column found!")

# ------------------------------------------
# Price Trend of Sample Symbols
# ------------------------------------------
sample_symbols = df["symbol"].unique()[:5]

plt.figure(figsize=(12,6))
for sym in sample_symbols:
    coin = df[df["symbol"] == sym].sort_values("date")
    plt.plot(coin["date"], coin["close"], label=sym)

plt.title("Closing Price Trend (Sample Cryptocurrencies)")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.show()

# ------------------------------------------
# Log Return Distribution
# ------------------------------------------
df = df.sort_values(["symbol", "date"])
df["log_return"] = df.groupby("symbol")["close"].apply(lambda x: np.log(x) - np.log(x.shift(1)))

plt.figure(figsize=(10,5))
sns.histplot(df["log_return"].dropna(), bins=100, kde=True)
plt.title("Distribution of Daily Log Returns")
plt.xlabel("Log Return")
plt.ylabel("Frequency")
plt.show()

# ------------------------------------------
# Rolling Volatility (Example Symbol)
# ------------------------------------------
example_symbol = df["symbol"].unique()[0]
example_coin = df[df["symbol"] == example_symbol].sort_values("date").copy()

example_coin["rolling_vol_14"] = example_coin["log_return"].rolling(14).std()

plt.figure(figsize=(12,5))
plt.plot(example_coin["date"], example_coin["rolling_vol_14"])
plt.title(f"14-Day Rolling Volatility ({example_symbol})")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.show()

# ------------------------------------------
# Correlation Heatmap
# ------------------------------------------
num_cols = ["open", "high", "low", "close", "volume", "market_cap"]

plt.figure(figsize=(8,5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# ------------------------------------------
# Volume vs Market Cap Scatter Plot
# ------------------------------------------
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="market_cap", y="volume", alpha=0.3)
plt.title("Volume vs Market Cap")
plt.xlabel("Market Cap")
plt.ylabel("Volume")
plt.show()

print("\nEDA Completed Successfully!")

NameError: name 'df' is not defined

In [2]:
import pandas as pd
import numpy as np

# Assuming df from previous step is already loaded and cleaned
# If not loaded, run:
# df = pd.read_csv("data/dataset.csv", parse_dates=["date"])

# Ensure data is sorted by symbol and date
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

# ------------------------------------------
# 1. Log Returns
# ------------------------------------------
df["log_return"] = df.groupby("symbol")["close"].apply(
    lambda x: np.log(x) - np.log(x.shift(1))
)

# ------------------------------------------
# 2. Lag Features (previous days' returns)
# ------------------------------------------
df["log_return_1"] = df.groupby("symbol")["log_return"].shift(1)
df["log_return_2"] = df.groupby("symbol")["log_return"].shift(2)
df["log_return_3"] = df.groupby("symbol")["log_return"].shift(3)

# ------------------------------------------
# 3. Rolling Volatility Features
# ------------------------------------------
df["vol_7"]  = df.groupby("symbol")["log_return"].rolling(7).std().reset_index(level=0, drop=True)
df["vol_14"] = df.groupby("symbol")["log_return"].rolling(14).std().reset_index(level=0, drop=True)
df["vol_30"] = df.groupby("symbol")["log_return"].rolling(30).std().reset_index(level=0, drop=True)

# ------------------------------------------
# 4. Moving Averages (MA) and Moving Std
# ------------------------------------------
df["ma_7"]  = df.groupby("symbol")["close"].rolling(7).mean().reset_index(level=0, drop=True)
df["ma_14"] = df.groupby("symbol")["close"].rolling(14).mean().reset_index(level=0, drop=True)
df["ma_30"] = df.groupby("symbol")["close"].rolling(30).mean().reset_index(level=0, drop=True)

df["std_7"]  = df.groupby("symbol")["close"].rolling(7).std().reset_index(level=0, drop=True)
df["std_14"] = df.groupby("symbol")["close"].rolling(14).std().reset_index(level=0, drop=True)

# ------------------------------------------
# 5. Liquidity Ratio
# ------------------------------------------
df["liquidity_ratio"] = df["volume"] / (df["market_cap"] + 1e-9)

# ------------------------------------------
# 6. Technical Indicators
# ------------------------------------------

# ATR (Average True Range)
df["tr"] = df["high"] - df["low"]  # simplified TR
df["atr_14"] = df.groupby("symbol")["tr"].rolling(14).mean().reset_index(level=0, drop=True)

# Bollinger Band Width
rolling_mean_20 = df.groupby("symbol")["close"].rolling(20).mean().reset_index(level=0, drop=True)
rolling_std_20  = df.groupby("symbol")["close"].rolling(20).std().reset_index(level=0, drop=True)
df["bb_width"] = (rolling_std_20 * 2) / rolling_mean_20

# ------------------------------------------
# 7. Daily Range %
# ------------------------------------------
df["range_pct"] = (df["high"] - df["low"]) / df["open"]

# ------------------------------------------
# 8. Calendar Features
# ------------------------------------------
df["day_of_week"] = df["date"].dt.dayofweek
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# ------------------------------------------
# 9. Target Variable: Future Volatility (Next-Day)
# ------------------------------------------
df["future_volatility"] = df.groupby("symbol")["vol_14"].shift(-1)

# Safe function to create volatility label
def safe_qcut(x):
    x = x.copy()
    if x.dropna().nunique() < 2:  # Not enough values to split
        return pd.Series(["normal"] * len(x), index=x.index)
    return pd.qcut(x, q=[0, 0.8, 1], labels=["normal", "high"])

# Create classification target
df["vol_label"] = df.groupby("symbol")["future_volatility"].transform(safe_qcut)

# ------------------------------------------
# 10. Final Cleanup
# ------------------------------------------
df = df.dropna().reset_index(drop=True)

print("Feature engineering completed!")
print("\nFinal columns:")
print(df.columns.tolist())

df.head()

NameError: name 'df' is not defined